In [ ]:
import os
import csv
import json
import math

# 配置路径
ROOT_DIR_EVAL = '/home/students/wcao/tuh_eeg_seizure/edf/eval/'
OUTPUT_FILE_EVAL = 'eval_segments.json'
SAMPLING_RATE = 250.0

def calculate_merged_seizure_duration(intervals):
    """
    计算所有 seizure 时间段的总时长（处理重叠）。
    intervals: list of tuple (start, stop)
    """
    if not intervals:
        return 0.0

    # 1. 按开始时间排序
    sorted_intervals = sorted(intervals, key=lambda x: x[0])

    merged = []
    for start, stop in sorted_intervals:
        if not merged:
            merged.append([start, stop])
        else:
            prev_start, prev_stop = merged[-1]
            if start < prev_stop:
                # 发生重叠或连接，更新结束时间为两者的最大值
                merged[-1][1] = max(prev_stop, stop)
            else:
                # 不重叠，添加新区间
                merged.append([start, stop])

    # 2. 计算合并后的总时长
    total_duration = sum(stop - start for start, stop in merged)
    return total_duration

def parse_csv_file(file_path):
    events = []
    seizure_intervals = [] # 用于计算总 seizure 时长
    total_duration_sec = 0.0
    
    with open(file_path, 'r', encoding='utf-8') as f:
        # 读取文件内容
        lines = f.readlines()
        
    # 解析 Header 获取 total_duration
    # Header 示例: # duration = 811.00 secs
    for line in lines:
        if line.startswith('#'):
            if 'duration' in line:
                try:
                    # 分割字符串找到数值部分
                    parts = line.split('=')
                    if len(parts) > 1:
                        dur_str = parts[1].strip().split()[0]
                        total_duration_sec = float(dur_str)
                except ValueError:
                    print(f"Warning: Could not parse duration in {file_path}")
    
    # 解析 CSV 数据部分
    # 找到非注释行的开始
    csv_lines = [line for line in lines if not line.startswith('#') and line.strip()]
    
    reader = csv.reader(csv_lines)
    
    # CSV 结构: channel, start_time, stop_time, label, confidence
    for row in reader:
        if len(row) < 4:
            continue
            
        channel_raw = row[0].strip()
        try:
            start_time = float(row[1].strip())
            stop_time = float(row[2].strip())
        except ValueError:
            continue # 跳过无法解析时间的行
            
        label = row[3].strip()
        
        # 解析 channel pair (例如 "T3-T5" -> ["T3", "T5"])
        # 有些特殊的 channel 名字可能不带 '-'，做一下兼容
        if '-' in channel_raw:
            pair = channel_raw.split('-')
        else:
            pair = [channel_raw] # 极端情况

        event_obj = {
            "pair": pair,
            "start_time": start_time,
            "stop_time": stop_time,
            "label": label
        }
        events.append(event_obj)
        
        # 收集非背景波的时间段用于计算时长
        if label != 'bckg':
            seizure_intervals.append((start_time, stop_time))
            
    return total_duration_sec, events, seizure_intervals

def stat_segments(root_dir, output_file, sampling_rate=250.0):
    metadata_list = []
    
    print(f"Scanning directory: {root_dir} ...")
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            # 过滤文件：只处理 .csv 且忽略 _bi.csv
            if file.endswith('.csv') and not file.endswith('_bi.csv'):
                full_path = os.path.join(root, file)
                
                # 解析路径结构
                # 假设结构是: .../patient/session/montage/filename.csv
                # 例如: .../aaaaaqvx/s002_2015/01_tcp_ar/aaaaaqvx_s002_t001.csv
                
                # 获取相对路径 (相对于 eval 目录)
                rel_path = os.path.relpath(full_path, root_dir)
                path_parts = rel_path.split(os.sep)
                
                # 确保路径深度足够提取信息
                if len(path_parts) >= 3:
                    patient = path_parts[-3] # aaaaaqvx
                    session = path_parts[-2] # s002_2015
                    montage = path_parts[-1] # 01_tcp_ar (这里注意，通常 csv 在 montage 文件夹内)
                    # 如果 csv 直接在 session 下，可能需要调整索引，但根据你的示例，csv在montage文件夹内
                    # rel_path 示例: aaaaaqvx/s002_2015/01_tcp_ar/aaaaaqvx_s002_t001.csv
                    # path_parts: ['aaaaaqvx', 's002_2015', '01_tcp_ar', 'aaaaaqvx_s002_t001.csv']
                    
                    if len(path_parts) == 4:
                        patient = path_parts[0]
                        session = path_parts[1]
                        montage = path_parts[2]
                        filename = path_parts[3]
                    else:
                        # 简单的回退策略，或者根据实际情况调整
                        patient = path_parts[0]
                        session = path_parts[1] if len(path_parts) > 1 else "unknown"
                        montage = path_parts[2] if len(path_parts) > 2 else "unknown"
                        filename = file

                    segment_name = os.path.splitext(filename)[0]
                    file_path = f"{patient}/{session}/{montage}/{segment_name}" # 构建你要求的 file_path 格式

                    # 解析 CSV 内容
                    total_dur, events, sz_intervals = parse_csv_file(full_path)
                    
                    # 计算 Seizure 总时长 (合并重叠)
                    seizure_dur = calculate_merged_seizure_duration(sz_intervals)
                    
                    # 计算百分比
                    percentage = 0.0
                    if total_dur > 0:
                        percentage = (seizure_dur / total_dur) * 100
                    
                    # 计算数据点
                    data_points = int(total_dur * sampling_rate)
                    
                    # 构建元数据对象
                    meta = {
                        "patient": patient,
                        "session": session,
                        "segment": segment_name,
                        "montage": montage,
                        "total_duration_sec": total_dur,
                        "seizure_duration_sec": seizure_dur,
                        "percentage": percentage,
                        "data_points": data_points,
                        "events": events,
                        "file_path": file_path
                    }
                    
                    metadata_list.append(meta)

    # 输出 JSON 文件
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(metadata_list, f, indent=4)
        
    print(f"Successfully processed {len(metadata_list)} files.")
    print(f"Output saved to: {os.path.abspath(output_file)}")
    
stat_segments(ROOT_DIR_EVAL, OUTPUT_FILE_EVAL, SAMPLING_RATE)

Scanning directory: /home/students/wcao/tuh_eeg_seizure/edf/eval/ ...
Successfully processed 865 files.
Output saved to: /home/students/wcao/eeg_with_wb/eval_segments.json


In [8]:
ROOT_DIR_TRAIN = '/home/students/wcao/tuh_eeg_seizure/edf/train/'
OUTPUT_FILE_TRAIN = 'train_segments.json'

stat_segments(ROOT_DIR_TRAIN, OUTPUT_FILE_TRAIN, SAMPLING_RATE)

Scanning directory: /home/students/wcao/tuh_eeg_seizure/edf/train/ ...
Successfully processed 4664 files.
Output saved to: /home/students/wcao/eeg_with_wb/train_segments.json


In [25]:
import json
from collections import Counter, defaultdict

class EEGSegment:
    """
    数据模型类：存储单个 EEG 片段的信息
    """
    def __init__(self, data):
        self.patient = data.get('patient', 'unknown')
        self.session = data.get('session', 'unknown')
        self.segment_name = data.get('segment', 'unknown')
        self.total_duration = data.get('total_duration_sec', 0.0)
        self.seizure_duration = data.get('seizure_duration_sec', 0.0)
        self.events = data.get('events', [])
        self.file_path = data.get('file_path', '') # e.g. "aaaa/s001/01_tcp/seg_name"
        
        # 唯一标识符：确保 session 统计时不会混淆（虽然 patient+session 通常唯一）
        self.session_id = f"{self.patient}_{self.session}"

    @property
    def has_seizure(self):
        """判断该片段是否包含 seizure 事件"""
        # 只要 seizure 持续时间大于 0 即视为有 seizure
        return self.seizure_duration > 0.0

    def __repr__(self):
        return f"<Segment: {self.segment_name}, Seizure: {self.has_seizure}>"


class EEGDataset:
    """
    数据集管理类：读取 JSON 并进行统计分析
    """
    def __init__(self, json_path):
        self.segments = []
        self.json_path = json_path
        self._load_data()

    def _load_data(self):
        """读取 JSON 并转换为对象列表"""
        print(f"正在读取文件: {self.json_path} ...")
        try:
            with open(self.json_path, 'r', encoding='utf-8') as f:
                raw_list = json.load(f)
                self.segments = [EEGSegment(item) for item in raw_list]
            print(f"成功加载 {len(self.segments)} 个片段数据。\n")
        except FileNotFoundError:
            print(f"错误: 找不到文件 {self.json_path}")
            self.segments = []
        except json.JSONDecodeError:
            print(f"错误: JSON 格式解析失败")
            self.segments = []

    def get_statistics(self):
        """执行统计逻辑"""
        if not self.segments:
            print("没有数据可统计。")
            return

        # 1. 基础集合 (用于去重统计)
        all_patients = set()
        all_sessions = set()
        
        # 2. Seizure 相关集合
        seizure_patients = set()
        seizure_sessions = set()
        seizure_segment_count = 0

        # 3. 聚合统计 (每个病人的片段数)
        patient_segment_counter = Counter()

        # 遍历列表进行统计
        for seg in self.segments:
            # 基础统计
            all_patients.add(seg.patient)
            all_sessions.add(seg.session_id)
            patient_segment_counter[seg.patient] += 1

            # Seizure 统计
            if seg.has_seizure:
                seizure_segment_count += 1
                seizure_patients.add(seg.patient)
                seizure_sessions.add(seg.session_id)

        # 4. 计算 Top 5 病人 (拥有最多 Segments)
        top_5_patients = patient_segment_counter.most_common(5)

        # --- 打印报告 ---
        print("=" * 40)
        print("EEG 数据集统计报告")
        print("=" * 40)
        
        print(f"{'统计维度':<20} | {'总数':<10} | {'含 Seizure 数量':<15} | {'Seizure 占比':<10}")
        print("-" * 65)
        
        # Segments
        total_segs = len(self.segments)
        seg_pct = (seizure_segment_count / total_segs * 100) if total_segs > 0 else 0
        print(f"{'Segments (片段)':<20} | {total_segs:<10} | {seizure_segment_count:<15} | {seg_pct:.2f}%")
        
        # Sessions
        total_sess = len(all_sessions)
        sess_count = len(seizure_sessions)
        sess_pct = (sess_count / total_sess * 100) if total_sess > 0 else 0
        print(f"{'Sessions (会话)':<20} | {total_sess:<10} | {sess_count:<15} | {sess_pct:.2f}%")
        
        # Patients
        total_pats = len(all_patients)
        pat_count = len(seizure_patients)
        pat_pct = (pat_count / total_pats * 100) if total_pats > 0 else 0
        print(f"{'Patients (病人)':<20} | {total_pats:<10} | {pat_count:<15} | {pat_pct:.2f}%")
        
        print("=" * 40)
        print("\n拥有最多 Segments 的前 5 位病人:")
        print("-" * 30)
        for rank, (patient, count) in enumerate(top_5_patients, 1):
            is_seizure_pat = "是" if patient in seizure_patients else "否"
            print(f"{rank}. 病人ID: {patient:<12} 片段数: {count:<5} (是否有癫痫: {is_seizure_pat})")
        print("-" * 30)


json_file_eval = 'eval_segments.json'

dataset_eval = EEGDataset(json_file_eval)
dataset_eval.get_statistics()


正在读取文件: eval_segments.json ...
成功加载 865 个片段数据。

EEG 数据集统计报告
统计维度                 | 总数         | 含 Seizure 数量    | Seizure 占比
-----------------------------------------------------------------
Segments (片段)        | 865        | 195             | 22.54%
Sessions (会话)        | 126        | 63              | 50.00%
Patients (病人)        | 43         | 34              | 79.07%

拥有最多 Segments 的前 5 位病人:
------------------------------
1. 病人ID: aaaaaqek     片段数: 53    (是否有癫痫: 是)
2. 病人ID: aaaaajru     片段数: 49    (是否有癫痫: 是)
3. 病人ID: aaaaahlk     片段数: 46    (是否有癫痫: 是)
4. 病人ID: aaaaardf     片段数: 45    (是否有癫痫: 是)
5. 病人ID: aaaaaarq     片段数: 43    (是否有癫痫: 是)
------------------------------


In [10]:
json_file_train = 'train_segments.json'

dataset_train = EEGDataset(json_file_train)
dataset_train.get_statistics()

正在读取文件: train_segments.json ...
成功加载 4664 个片段数据。

EEG 数据集统计报告
统计维度                 | 总数         | 含 Seizure 数量    | Seizure 占比
-----------------------------------------------------------------
Segments (片段)        | 4664       | 872             | 18.70%
Sessions (会话)        | 1174       | 352             | 29.98%
Patients (病人)        | 579        | 208             | 35.92%

拥有最多 Segments 的前 5 位病人:
------------------------------
1. 病人ID: aaaaanme     片段数: 97    (是否有癫痫: 是)
2. 病人ID: aaaaaoya     片段数: 96    (是否有癫痫: 是)
3. 病人ID: aaaaapks     片段数: 94    (是否有癫痫: 是)
4. 病人ID: aaaaamhb     片段数: 83    (是否有癫痫: 是)
5. 病人ID: aaaaagxr     片段数: 78    (是否有癫痫: 否)
------------------------------


In [11]:
import os
import mne
import re

# =================配置区域=================
MIN_SFREQ = 250.0

# 标准通道列表 (必须包含的通道)
STANDARD_CHANNELS = {
    'FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2', 
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'CZ'
}
# =========================================

def normalize_channel_name(ch_name):
    """
    使用正则处理通道名称。
    目标格式: EEG FP1-REF -> FP1
    逻辑: 
    1. 去除 'EEG ' 前缀 (忽略大小写)
    2. 去除 '-REF' 后缀 (或其他引用后缀，这里主要针对 -REF)
    3. 或者直接提取中间的字母数字部分
    """
    # 方法 A: 严格匹配你的示例 'EEG FP1-REF'
    # match = re.search(r'EEG\s+([A-Z0-9]+)-REF', ch_name, re.IGNORECASE)
    # if match:
    #     return match.group(1).upper()
    
    # 方法 B: 更鲁棒的方法 (TUH 数据集通用)
    # 1. 去掉 'EEG '
    clean = ch_name.upper().replace('EEG', '').strip()
    # 2. 去掉 '-REF' 或其他后缀，通常以 '-' 分隔
    if '-' in clean:
        clean = clean.split('-')[0]
    return clean.strip()

def check_edf_files(root_dir):
    print(f"开始扫描目录: {root_dir} ...")
    
    total_files = 0
    failed_files = [] # 存储 (file_path, reason)
    
    # 统计信息
    sfreq_counter = {}
    
    for root, _, files in os.walk(root_dir):
        for file in files:
            if file.endswith('.edf'):
                total_files += 1
                file_path = os.path.join(root, file)
                
                try:
                    # 读取 header，preload=False 不加载数据，速度很快
                    # verbose=False 静默模式，不打印 MNE 的 info
                    raw = mne.io.read_raw_edf(file_path, preload=False, verbose='error')
                    info = raw.info
                    
                    # 1. 获取基本信息
                    sfreq = info['sfreq']
                    low_pass = info['lowpass']
                    high_pass = info['highpass']
                    ch_names = info['ch_names']
                    
                    # 记录采样率分布 (仅供统计查看)
                    sfreq_counter[sfreq] = sfreq_counter.get(sfreq, 0) + 1
                    
                    # --- 校验逻辑 ---
                    failure_reasons = []
                    
                    # 检查 1: 采样率 >= 250
                    if sfreq < MIN_SFREQ:
                        failure_reasons.append(f"Low Sampling Rate: {sfreq}Hz")
                    
                    # 检查 2: 通道完整性
                    # 清洗当前文件的所有通道名
                    current_clean_channels = set()
                    for ch in ch_names:
                        cleaned = normalize_channel_name(ch)
                        current_clean_channels.add(cleaned)
                    
                    # 判断是否包含所有标准通道 (Standard 必须是 Current 的子集)
                    missing_channels = STANDARD_CHANNELS - current_clean_channels
                    
                    if missing_channels:
                        failure_reasons.append(f"Missing Channels: {list(missing_channels)}")
                    
                    # 如果有任何失败原因，记录下来
                    if failure_reasons:
                        failed_files.append({
                            "path": file_path,
                            "reasons": failure_reasons,
                            "sfreq": sfreq,
                            "lp": low_pass,
                            "hp": high_pass,
                            "n_ch": len(ch_names)
                        })
                        
                except Exception as e:
                    # 记录读取错误的文件（可能是文件损坏）
                    failed_files.append({
                        "path": file_path,
                        "reasons": [f"Read Error: {str(e)}"],
                        "sfreq": "N/A",
                        "lp": "N/A",
                        "hp": "N/A",
                        "n_ch": "N/A"
                    })

    # ================= 打印报告 =================
    print("\n" + "="*60)
    print("扫描完成报告")
    print("="*60)
    print(f"扫描总文件数: {total_files}")
    print(f"不合格文件数: {len(failed_files)}")
    print(f"采样率分布: {sfreq_counter}")
    
    if len(failed_files) > 0:
        print("\n[不合格文件列表]:")
        print("-" * 60)
        for item in failed_files:
            print(f"文件: {item['path']}")
            print(f"  -> 原因: {', '.join(item['reasons'])}")
            if "Read Error" not in item['reasons'][0]:
                print(f"  -> Info: sfreq={item['sfreq']}, LP={item['lp']}, HP={item['hp']}, Chans={item['n_ch']}")
            print("-" * 60)
    else:
        print("\n完美！所有 EDF 文件均满足采样率 >= 250Hz 且包含所有标准通道。")

check_edf_files(ROOT_DIR_EVAL)

开始扫描目录: /home/students/wcao/tuh_eeg_seizure/edf/eval/ ...

扫描完成报告
扫描总文件数: 865
不合格文件数: 0
采样率分布: {256.0: 831, 1000.0: 18, 250.0: 16}

完美！所有 EDF 文件均满足采样率 >= 250Hz 且包含所有标准通道。


In [12]:
check_edf_files(ROOT_DIR_TRAIN)

开始扫描目录: /home/students/wcao/tuh_eeg_seizure/edf/train/ ...

扫描完成报告
扫描总文件数: 4664
不合格文件数: 0
采样率分布: {250.0: 930, 256.0: 3013, 400.0: 580, 1000.0: 47, 512.0: 94}

完美！所有 EDF 文件均满足采样率 >= 250Hz 且包含所有标准通道。


In [15]:
import os
import shutil
import uuid
import h5py
import mne
import numpy as np
import warnings

# ================= 配置区域 =================
# 输入目录 (EDF 所在)
SOURCE_ROOT_EVAL = '/home/students/wcao/tuh_eeg_seizure/edf/eval/'
# 输出目录 (H5 存储位置)
TARGET_ROOT_EVAL = '/home/students/wcao/eeg_test_seizure_h5'
# 临时目录
TMP_DIR = '/tmp'

# 目标采样率
TARGET_SFREQ = 250.0

# 标准通道顺序 (严格有序)
STANDARD_CHANNELS = [
    'FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'CZ'
]
# ===========================================

def normalize_channel_name(ch_name):
    """
    清洗通道名称，例如 'EEG FP1-REF' -> 'FP1'
    """
    clean = ch_name.upper().replace('EEG', '').strip()
    if '-' in clean:
        clean = clean.split('-')[0]
    return clean.strip()

def process_single_file(edf_path, output_root, source_root):
    """
    处理单个文件的完整流程：Copy -> Load -> Preprocess -> Save -> Move
    """
    unique_id = str(uuid.uuid4())[:8]
    tmp_edf_path = os.path.join(TMP_DIR, f"temp_{unique_id}.edf")
    tmp_h5_path = os.path.join(TMP_DIR, f"temp_{unique_id}.h5")
    
    try:
        # 1. 安全复制 EDF 到 /tmp
        shutil.copyfile(edf_path, tmp_edf_path)
        
        # 2. MNE 读取 (preload=True 以便进行重采样和修改)
        # 忽略读取过程中的一些非致命警告
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            raw = mne.io.read_raw_edf(tmp_edf_path, preload=True, verbose='error')
        
        # 3. 通道匹配与重排
        original_names = raw.ch_names
        # 建立映射: 标准名 -> 原始名
        # 例如: {'FP1': 'EEG FP1-REF', 'CZ': 'EEG CZ-REF', ...}
        std_to_orig_map = {}
        
        # 遍历原始通道，尝试归一化并匹配
        for orig_ch in original_names:
            norm_name = normalize_channel_name(orig_ch)
            if norm_name in STANDARD_CHANNELS:
                std_to_orig_map[norm_name] = orig_ch
        
        # 检查完整性
        missing_channels = [ch for ch in STANDARD_CHANNELS if ch not in std_to_orig_map]
        if missing_channels:
            print(f"Skipping {os.path.basename(edf_path)}: Missing channels {missing_channels}")
            return False

        # 4. 提取并重排数据
        # 我们不能简单用 raw.pick_channels，因为那不保证顺序。
        # 我们按照 STANDARD_CHANNELS 的顺序，手动挑选原始通道名。
        ordered_orig_names = [std_to_orig_map[std_ch] for std_ch in STANDARD_CHANNELS]
        
        # Pick 仅保留需要的通道 (这一步 MNE 会自动处理内存中的数据剔除)
        raw.pick_channels(ordered_orig_names)
        
        # Reorder 强制按列表顺序排列
        raw.reorder_channels(ordered_orig_names)
        
        # 5. 重采样 (如果需要)
        if raw.info['sfreq'] != TARGET_SFREQ:
            # print(f"  Resampling {raw.info['sfreq']} -> {TARGET_SFREQ} Hz")
            raw.resample(TARGET_SFREQ, npad="auto")
            
        # 6. 获取 numpy 矩阵 [n_channels, n_times]
        data = raw.get_data()
        
        # 再次确认形状
        if data.shape[0] != 17:
            print(f"Error: Output channels mismatch. Expected 17, got {data.shape[0]}")
            return False

        # 7. 写入本地临时 H5
        with h5py.File(tmp_h5_path, 'w') as f:
            # 兼容你提供的 load_data_from_h5 接口
            f.create_dataset('eeg', data=data, compression="gzip", compression_opts=4)
            # 可选：写入通道名称元数据，方便日后查验
            # f.create_dataset('ch_names', data=np.array(STANDARD_CHANNELS, dtype='S'))
            # f.attrs['sfreq'] = TARGET_SFREQ
            
        # 8. 计算目标路径并移动
        # 计算相对路径: aaaaaqvx/s002_2015/01_tcp_ar/file.edf
        rel_path = os.path.relpath(edf_path, source_root)
        # 替换扩展名 .edf -> .h5
        rel_path_h5 = os.path.splitext(rel_path)[0] + '.h5'
        
        final_h5_path = os.path.join(output_root, rel_path_h5)
        
        # 确保目标子目录存在
        os.makedirs(os.path.dirname(final_h5_path), exist_ok=True)
        
        # 移动文件 (Move 是原子操作，或者在跨文件系统时是 Copy+Delete)
        shutil.move(tmp_h5_path, final_h5_path)
        
        return True

    except Exception as e:
        print(f"Failed to process {os.path.basename(edf_path)}: {e}")
        return False
        
    finally:
        # 清理临时文件
        if os.path.exists(tmp_edf_path):
            try: os.remove(tmp_edf_path)
            except: pass
        if os.path.exists(tmp_h5_path):
            try: os.remove(tmp_h5_path)
            except: pass

def convert_edf_to_h5(source_root, target_root):
    print(f"Start converting EDF to H5...")
    print(f"Source: {source_root}")
    print(f"Target: {target_root}")
    
    success_count = 0
    fail_count = 0
    
    # 遍历源目录
    for root, dirs, files in os.walk(source_root):
        for file in files:
            if file.endswith('.edf'):
                edf_full_path = os.path.join(root, file)
                
                # 执行转换
                if process_single_file(edf_full_path, target_root, source_root):
                    success_count += 1
                    if success_count % 10 == 0:
                        print(f"Processed {success_count} files...", end='\r')
                else:
                    fail_count += 1
    
    print(f"\nconversion finished!")
    print(f"Success: {success_count}")
    print(f"Failed/Skipped: {fail_count}")

convert_edf_to_h5(SOURCE_ROOT_EVAL, TARGET_ROOT_EVAL)

Start converting EDF to H5...
Source: /home/students/wcao/tuh_eeg_seizure/edf/eval/
Target: /home/students/wcao/eeg_test_seizure_h5
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst.pick(...).
NOTE: pick_channels() is a legacy function. New code should use inst

In [ ]:
import os
import shutil
import uuid
import h5py
import mne
import numpy as np
import warnings
import json

# ================= 配置区域 =================
# JSON 路径 (请确保这里指向你的 train_segments.json)
JSON_FILE_TRAIN = 'train_segments.json'

# 输入目录 (EDF 训练集根目录)
SOURCE_ROOT_TRAIN = '/home/students/wcao/tuh_eeg_seizure/edf/train/'
# 输出目录 (H5 存储位置)
TARGET_ROOT_TRAIN = '/home/students/wcao/eeg_train_seizure_h5'
# 临时目录
TMP_DIR = '/tmp'

# 目标采样率
TARGET_SFREQ = 250.0

# 标准通道顺序
STANDARD_CHANNELS = [
    'FP1', 'FP2', 'F3', 'F4', 'C3', 'C4', 'P3', 'P4', 'O1', 'O2',
    'F7', 'F8', 'T3', 'T4', 'T5', 'T6', 'CZ'
]
# ===========================================

# --- 1. 定义数据类 (复用之前的逻辑) ---
class EEGSegment:
    def __init__(self, data):
        self.patient = data.get('patient', 'unknown')
        self.session = data.get('session', 'unknown')
        self.segment_name = data.get('segment', 'unknown')
        self.seizure_duration = data.get('seizure_duration_sec', 0.0)
        self.file_path = data.get('file_path', '') # e.g. "aaaa/s001/01_tcp/seg_name"

    @property
    def has_seizure(self):
        return self.seizure_duration > 0.0

class EEGDataset:
    def __init__(self, json_path):
        self.segments = []
        self.json_path = json_path
        self._load_data()

    def _load_data(self):
        print(f"Loading metadata from: {self.json_path} ...")
        try:
            with open(self.json_path, 'r', encoding='utf-8') as f:
                raw_list = json.load(f)
                self.segments = [EEGSegment(item) for item in raw_list]
            print(f"Loaded {len(self.segments)} segments.")
        except Exception as e:
            print(f"Error loading JSON: {e}")
            self.segments = []

# --- 2. 辅助函数 ---
def normalize_channel_name(ch_name):
    clean = ch_name.upper().replace('EEG', '').strip()
    if '-' in clean:
        clean = clean.split('-')[0]
    return clean.strip()

def process_single_file(edf_path, h5_target_path):
    """
    转换单个文件: EDF -> Tmp -> H5 -> Move to Target
    """
    unique_id = str(uuid.uuid4())[:8]
    tmp_edf_path = os.path.join(TMP_DIR, f"temp_{unique_id}.edf")
    tmp_h5_path = os.path.join(TMP_DIR, f"temp_{unique_id}.h5")
    
    try:
        # 1. 复制到 /tmp
        shutil.copyfile(edf_path, tmp_edf_path)
        
        # 2. MNE 读取
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            raw = mne.io.read_raw_edf(tmp_edf_path, preload=True, verbose='error')
        
        # 3. 通道筛选与重排
        std_to_orig_map = {}
        for orig_ch in raw.ch_names:
            norm_name = normalize_channel_name(orig_ch)
            if norm_name in STANDARD_CHANNELS:
                std_to_orig_map[norm_name] = orig_ch
        
        # 检查通道完整性
        missing = [ch for ch in STANDARD_CHANNELS if ch not in std_to_orig_map]
        if missing:
            return f"MISSING_CHANNELS: {missing}"

        ordered_names = [std_to_orig_map[ch] for ch in STANDARD_CHANNELS]
        raw.pick_channels(ordered_names)
        raw.reorder_channels(ordered_names)
        
        # 4. 重采样
        if raw.info['sfreq'] != TARGET_SFREQ:
            raw.resample(TARGET_SFREQ, npad="auto")
            
        # 5. 写入 H5
        data = raw.get_data()
        if data.shape[0] != 17:
            return f"SHAPE_MISMATCH: {data.shape}"

        with h5py.File(tmp_h5_path, 'w') as f:
            f.create_dataset('eeg', data=data, compression="gzip", compression_opts=4)
            
        # 6. 移动到目标位置
        os.makedirs(os.path.dirname(h5_target_path), exist_ok=True)
        shutil.move(tmp_h5_path, h5_target_path)
        
        return "SUCCESS"

    except Exception as e:
        return f"ERROR: {str(e)}"
        
    finally:
        # 清理
        for p in [tmp_edf_path, tmp_h5_path]:
            if os.path.exists(p):
                try: os.remove(p)
                except: pass

# --- 3. 主流程 ---
def convert_and_check(json_file, source_root, target_root):
    # 1. 加载数据集元数据
    dataset = EEGDataset(json_file)
    if not dataset.segments:
        return

    print(f"Target Directory: {target_root}")
    
    stats = {
        "processed": 0,
        "skipped_seizure": 0,
        "skipped_collision": 0, # 本来应该是背景，但文件已存在
        "failed": 0
    }
    collision_list = []

    # 2. 遍历处理
    for seg in dataset.segments:
        # 构建完整路径
        # JSON 中的 file_path 通常是 "aaaa/s001/01_tcp/seg_name" (无后缀)
        rel_path_base = seg.file_path
        
        # 源文件 (.edf)
        source_edf_path = os.path.join(source_root, rel_path_base + '.edf')
        # 目标文件 (.h5)
        target_h5_path = os.path.join(target_root, rel_path_base + '.h5')
        
        # --- 筛选逻辑 ---
        if seg.has_seizure:
            # 是癫痫片段 -> 跳过 (因为你之前已经处理过了)
            stats['skipped_seizure'] += 1
            continue
        
        # 是背景片段 -> 检查是否已存在
        if os.path.exists(target_h5_path):
            # 冲突！说明这个背景文件已经存在了
            stats['skipped_collision'] += 1
            collision_list.append(target_h5_path)
            continue
        
        # 检查源文件是否存在
        if not os.path.exists(source_edf_path):
            print(f"Source not found: {source_edf_path}")
            stats['failed'] += 1
            continue
            
        # --- 开始转换 ---
        res = process_single_file(source_edf_path, target_h5_path)
        
        if res == "SUCCESS":
            stats['processed'] += 1
            if stats['processed'] % 10 == 0:
                print(f"Converted {stats['processed']} background files...", end='\r')
        else:
            print(f"Failed {rel_path_base}: {res}")
            stats['failed'] += 1

    # 3. 报告
    print("\n" + "="*50)
    print("Background File Conversion Report")
    print("="*50)
    print(f"Converted (New Background):   {stats['processed']}")
    print(f"Skipped (Is Seizure):         {stats['skipped_seizure']}")
    print(f"Skipped (Collision/Exists):   {stats['skipped_collision']}")
    print(f"Failed (Errors):              {stats['failed']}")
    
    if collision_list:
        print("\n[WARNING] Collisions detected (Background files that already existed):")
        for f in collision_list[:10]:
            print(f)
        if len(collision_list) > 10:
            print(f"... and {len(collision_list)-10} more.")
            
convert_and_check(JSON_FILE_TRAIN, SOURCE_ROOT_TRAIN, TARGET_ROOT_TRAIN)

In [21]:
import os

def get_file_identifiers(root_dir, extension):
    """
    遍历目录，返回所有文件的相对路径ID集合 (去掉扩展名)
    例如: aaaaa/s001/01_tcp/file.edf -> aaaaa/s001/01_tcp/file
    """
    identifiers = set()
    print(f"正在扫描目录: {root_dir} (查找 *{extension}) ...")
    
    count = 0
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.endswith(extension):
                # 获取完整路径
                full_path = os.path.join(root, file)
                
                # 获取相对路径 (去掉前面的 /home/.../train/)
                rel_path = os.path.relpath(full_path, root_dir)
                
                # 去掉扩展名，得到 ID
                # os.path.splitext 处理 "path/to/file.edf" -> ("path/to/file", ".edf")
                file_id = os.path.splitext(rel_path)[0]
                
                identifiers.add(file_id)
                count += 1
                if count % 1000 == 0:
                    print(f"  已找到 {count} 个文件...", end='\r')
                    
    print(f"  扫描完成。共找到 {len(identifiers)} 个唯一 ID。")
    return identifiers

def verify_edf_h5(source_root, target_root):
    print("开始一一对应验证 (One-to-One Verification)\n")

    # 1. 获取 ID 集合
    edf_ids = get_file_identifiers(source_root, '.edf')
    h5_ids = get_file_identifiers(target_root, '.h5')

    # 2. 计算差异
    # 存在于 EDF 但不存在于 H5 (漏转的，或转换失败的)
    missing_h5 = edf_ids - h5_ids
    
    # 存在于 H5 但不存在于 EDF (可能是以前跑的旧文件，或者是文件名对应错误的)
    extra_h5 = h5_ids - edf_ids

    # 3. 打印报告
    print("\n" + "="*60)
    print("验证报告 (Verification Report)")
    print("="*60)
    print(f"源目录 (EDF): {len(edf_ids)}")
    print(f"目标目录 (H5) : {len(h5_ids)}")
    print("-" * 60)

    if not missing_h5 and not extra_h5:
        print("\n✅ 完美匹配！源文件和目标文件一一对应 (1:1 Match)。")
    else:
        # 报告缺失的 H5
        if missing_h5:
            print(f"\n❌ 缺失 H5 文件 (源目录有 EDF，但目标目录无 H5): 共 {len(missing_h5)} 个")
            print("列表如下 (前 50 个):")
            for i, fid in enumerate(sorted(list(missing_h5))):
                if i >= 50:
                    print(f"  ... 以及其他 {len(missing_h5) - 50} 个")
                    break
                print(f"  [MISSING] {fid}.edf")

        # 报告多余的 H5
        if extra_h5:
            print(f"\n⚠️  多余 H5 文件 (目标目录有 H5，但源目录无 EDF): 共 {len(extra_h5)} 个")
            print("列表如下 (前 50 个):")
            for i, fid in enumerate(sorted(list(extra_h5))):
                if i >= 50:
                    print(f"  ... 以及其他 {len(extra_h5) - 50} 个")
                    break
                print(f"  [EXTRA]   {fid}.h5")

    print("\n" + "="*60)

verify_edf_h5(SOURCE_ROOT_EVAL, TARGET_ROOT_EVAL)
verify_edf_h5(SOURCE_ROOT_TRAIN, TARGET_ROOT_TRAIN)

开始一一对应验证 (One-to-One Verification)

正在扫描目录: /home/students/wcao/tuh_eeg_seizure/edf/eval/ (查找 *.edf) ...
  扫描完成。共找到 865 个唯一 ID。
正在扫描目录: /home/students/wcao/eeg_test_seizure_h5 (查找 *.h5) ...
  扫描完成。共找到 865 个唯一 ID。

验证报告 (Verification Report)
源目录 (EDF): 865
目标目录 (H5) : 865
------------------------------------------------------------

✅ 完美匹配！源文件和目标文件一一对应 (1:1 Match)。

开始一一对应验证 (One-to-One Verification)

正在扫描目录: /home/students/wcao/tuh_eeg_seizure/edf/train/ (查找 *.edf) ...
  扫描完成。共找到 4664 个唯一 ID。
正在扫描目录: /home/students/wcao/eeg_train_seizure_h5 (查找 *.h5) ...
  扫描完成。共找到 4664 个唯一 ID。

验证报告 (Verification Report)
源目录 (EDF): 4664
目标目录 (H5) : 4664
------------------------------------------------------------

✅ 完美匹配！源文件和目标文件一一对应 (1:1 Match)。



In [28]:
def load_data_from_h5(train_path_h5, tmp_path, segment_file_path):
    h5_path = os.path.join(train_path_h5, segment_file_path + '.h5')
    tmp_file_path = os.path.join(tmp_path,  os.path.basename(segment_file_path) + '.h5')
    
    shutil.copyfile(h5_path, tmp_file_path)
    
    # Read the data from the temporary H5 file
    try:
        with h5py.File(tmp_file_path, 'r') as f:
            data = f['eeg'][:]
    except Exception as e:
        data = None
        
    # Clean up the temporary file
    try:
        os.remove(tmp_file_path)
    except Exception as e:
        print(f"Error deleting temporary file: {e}")
        
    return data

for seg in dataset_eval.segments[:15]:
    data = load_data_from_h5(TARGET_ROOT_EVAL, TMP_DIR, seg.file_path)
    if data is not None:
        print(f"Loaded data shape for {seg.file_path}: {data.shape}")
    else:
        print(f"Failed to load data for {seg.file_path}")

Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t001: (17, 150250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t009: (17, 150250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t005: (17, 150250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t003: (17, 150250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t000: (17, 153250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t006: (17, 75000)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t002: (17, 75000)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t008: (17, 75000)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t007: (17, 150250)
Loaded data shape for aaaaarpv/s003_2014/01_tcp_ar/aaaaarpv_s003_t004: (17, 75000)
Loaded data shape for aaaaarpv/s001_2014/01_tcp_ar/aaaaarpv_s001_t001: (17, 202500)
Loaded data shape for aaaaarpv/s001_2014/01_tcp_ar/aaaaarpv_s001_t003: (17, 1502